# 31. Dalitz-plot vetoes

**Objectives:**
- Build a `MassWindowVeto` (bounds in GeV, the Laura++ `addMassVeto` convention).
- Combine several vetoes with `CompositeVeto` (logical AND).
- Wrap an arbitrary accept/reject rule with `FunctionalVeto`.
- Apply a veto to a data sample with `veto.apply(sample)` and to an integration
  sample with `veto.apply(sample, for_integration=True)`, and see why they differ.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero. See `docs/backgrounds_and_vetoes.md` for the full
veto convention this notebook follows.


In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import numpy as np
import jax.numpy as jnp
from dalitzplotfitter import (
    CompositeVeto, DecayChannel, DecayModel, FunctionalVeto,
    MassWindowVeto, NonResonant, RealImag, Resonance, generate_toy,
)


## 1. A small model and a data sample

We use `B+ -> K+ pi+ pi-` (no identical daughters, so no symmetrization
subtleties). `pair=(0, 2)` is the `K+ pi-` system. `generate_toy` gives us an
unweighted data-like sample; `model.normalization_sample` is the deterministic
integration grid used for normalization.


In [2]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
model = DecayModel(
    channel,
    [
        Resonance("Kstar", (0, 2), RealImag(1.0, 0.0), mass=0.892, width=0.051, spin=1),
        NonResonant(RealImag(0.4, -0.2), name="NR"),
    ],
    normalization_method="square-dalitz", normalization_pair=(0, 2),
    normalization_resolution=60,
)
truth = {p.name: p.value for p in model.parameters}
data = generate_toy(model, 4000, parameters=truth, seed=31, include_momenta=False)
print(f"Generated {data.size} events; normalization grid has {model.normalization_sample.size} points.")


Generated 4000 events; normalization grid has 3600 points.


## 2. `MassWindowVeto`: reject one invariant-mass window

Bounds are an invariant mass **in GeV**, not a mass-squared in GeV² — this
mirrors Laura++'s `addMassVeto`, not the `s12`/`s13`/`s23` convention used
everywhere else in this package. Here we veto the `K+ pi-` (`pair=(0, 2)`)
system between 1.40 and 1.60 GeV.


In [3]:
kpi_veto = MassWindowVeto((0, 2), 1.40, 1.60)

accepted = kpi_veto.apply(data)
mass_kpi = np.asarray(jnp.sqrt(accepted.s13))
print(f"Kept {accepted.size} / {data.size} events after the K+pi- mass veto.")
assert not np.any((mass_kpi >= 1.40) & (mass_kpi <= 1.60))


Kept 3926 / 4000 events after the K+pi- mass veto.


## 3. `CompositeVeto`: combine vetoes with a logical AND

`CompositeVeto` accepts an event only if *every* sub-veto accepts it. Below we
add a second window on the `pi+ pi-` (`pair=(1, 2)`) system; the composite
result is a subset of either veto applied alone.


In [4]:
pipi_veto = MassWindowVeto((1, 2), 0.60, 0.80)
combined = CompositeVeto(kpi_veto, pipi_veto)

combined_accepted = combined.apply(data)
print(f"K+pi- veto alone keeps {kpi_veto.apply(data).size} events.")
print(f"pi+pi- veto alone keeps {pipi_veto.apply(data).size} events.")
print(f"Composite (AND) keeps {combined_accepted.size} events.")
assert combined_accepted.size <= min(kpi_veto.apply(data).size, pipi_veto.apply(data).size)


K+pi- veto alone keeps 3926 events.


pi+pi- veto alone keeps 3993 events.
Composite (AND) keeps 3919 events.


## 4. `FunctionalVeto`: an arbitrary accept/reject rule

`FunctionalVeto` wraps any callable returning a boolean mask of the right
shape. This is the escape hatch for a veto shape that a mass window or a
composite of mass windows cannot express — here, a diagonal cut mixing two
invariants.


In [5]:
def corner_cut(events):
    return (events["s12"] + events["s23"]) < 1.0 * (channel.parent_mass ** 2)

functional_veto = FunctionalVeto(corner_cut)
functional_accepted = functional_veto.apply(data)
print(f"Functional veto keeps {functional_accepted.size} / {data.size} events.")
assert np.all(np.asarray(corner_cut(functional_accepted.as_dict())))


Functional veto keeps 4000 / 4000 events.


## 5. Data selection vs. integration-sample selection

`veto.apply(sample)` simply drops rejected rows (`PhaseSpaceSample.take`) —
correct for an event sample, since dropped events should vanish, not be
reweighted. `veto.apply(sample, for_integration=True)` is different: it also
rescales the *kept* weights by `n_kept / n_total`
(`PhaseSpaceSample.select_for_integration`), so that
`mean(new_weights * integrand)` over the smaller sample is an unbiased
estimator of `mean(weights * integrand * mask)` over the original sample —
exactly what a normalization integral over the accepted region requires.


In [6]:
normalization_sample = model.normalization_sample
mask = np.asarray(combined.accept(normalization_sample.as_dict()))

# Plain apply(): rows are dropped, weights are untouched.
data_like = combined.apply(normalization_sample)
# for_integration=True: rows are dropped AND kept weights are rescaled.
integration_like = combined.apply(normalization_sample, for_integration=True)

original_weights = np.asarray(normalization_sample.weights)
estimate_from_full_sample = np.mean(original_weights * mask)
estimate_from_integration_sample = np.mean(np.asarray(integration_like.weights))
estimate_from_data_like_sample = np.mean(np.asarray(data_like.weights))

print("mean(weights * mask) on the full sample:      ", estimate_from_full_sample)
print("mean(new weights) on the integration sample:  ", estimate_from_integration_sample)
print("mean(weights) on the plain apply() sample:     ", estimate_from_data_like_sample)
np.testing.assert_allclose(estimate_from_full_sample, estimate_from_integration_sample, rtol=1e-10)


mean(weights * mask) on the full sample:       332.901222991353
mean(new weights) on the integration sample:   332.90122299135305
mean(weights) on the plain apply() sample:      363.60570472356517


Note that the plain `apply()` result's mean weight is biased high relative to
the full-sample estimate: it never applied the `n_kept/n_total` correction,
because that correction is meaningful only for a Monte Carlo integration
estimator, not for a selected event sample. Use `apply()` for data and toys;
use `apply(..., for_integration=True)` for any sample that feeds a
normalization integral (e.g. `vetoed_signal_pdf`, `VetoedDensity`, or a custom
`GridIntegrator`) — see the next lesson.

## Try it yourself

1. Build a `CompositeVeto` of three or more `MassWindowVeto` objects.
2. Write a `FunctionalVeto` that vetoes a disk in the `(s12, s23)` plane instead of a strip.
3. Check what happens if `for_integration=True` selects zero events (`select_for_integration` raises).

## Continue learning

Next: [Veto-aware densities and signal PDFs](tutorial_32_vetoed_density.ipynb).
Reference: [backgrounds and vetoes](../../docs/backgrounds_and_vetoes.md).

Return to [the course guide](TUTORIALS.md).
